# Data extraction 2025

Cleans the 2025/2026 Qualtrics export into the analysis-ready CSVs. The association and
valence tasks are extracted with the shared helpers in `data_cleaning_qualtrics`; the
2025-specific participant-ID scheme and the questionnaire parsing stay in the notebook.

In [2]:
import pandas as pd

from config import RESSERENO_ROOT
from data_cleaning_qualtrics import (
    get_associations,
    get_valences,
    associations_valences_df,
)

pd.set_option("display.max_columns", None)

path_data = RESSERENO_ROOT / "data"
path_out = RESSERENO_ROOT / "out" / "formatting" / "all_students_2025"

## Data extraction

In [3]:
# data_spring_2026 already re-includes every data_2026 response, so use it instead of data_2026
# (adding both would double-count the 2026 batch). its two Qualtrics header rows are dropped here;
# the 2025 header rows are removed together during trimming below.
raw_2025 = pd.read_csv(f"{path_data}/data_2025.csv")
raw_spring_2026 = pd.read_csv(f"{path_data}/data_2026.csv")[2:]
raw_data = pd.concat([raw_2025, raw_spring_2026], ignore_index=True)

In [4]:
DROP_COLUMNS = [
    "EndDate",
    "Status",
    "IPAddress",
    "RecordedDate",
    "ResponseId",
    "RecipientLastName",
    "RecipientFirstName",
    "RecipientEmail",
    "ExternalReference",
    "LocationLatitude",
    "LocationLongitude",
    "DistributionChannel",
    "UserLanguage",
]

# drop unused columns and the two leftover Qualtrics header rows
data = raw_data.drop(columns=DROP_COLUMNS)[2:]

# keep only the participations with >65% progress that gave consent (QID3 == "1")
progress = data["Progress"].values.astype(int)
data = data[(progress > 65) & (data["QID3"] == "1")].reset_index(drop=True)

## Participant ID creation

Participants come from several populations, told apart by the population question, the day they took
part (`StartDate`), or the session time slot.

Two independent survey revisions matter:
- **27 Feb 2026** — the cue word shown as *"Meditazione"* became *"ChatGPT"* (handled in the
  associations step via `CHATGPT_CUTOFF`).
- **4 May 2026** — the questionnaire was restructured: `Q20` became *studente/docente*, and the
  department question moved to `Q21` (asked to students only), so teachers have an empty `Q21`.
  Before this date the department was in `Q20`.

IDs are assigned per survey era, continuing the numbering of shared groups so the eras can be
analysed together.

In [5]:
# 2026-02-27: the cue word shown as "Meditazione" became "ChatGPT" (used in the associations step)
CHATGPT_CUTOFF = pd.Timestamp("2026-02-27")


def assign_id_by_question(df, q20_value, prefix, era_mask, start_idx=1, idx="Q20"):
    """Assign participant_id from the Q20 answer, within a single survey era."""
    mask = era_mask & (df[idx] == q20_value)
    df.loc[mask, "participant_id"] = [
        f"{prefix}_{i:03d}" for i in range(start_idx, start_idx + mask.sum())
    ]


def assign_id_by_date(df, dates, prefix):
    """Assign participant_id to everyone who took part on the given StartDate day(s)."""
    if isinstance(dates, str):
        dates = [dates]
    days = {pd.to_datetime(d).date() for d in dates}
    mask = df["StartDate"].dt.date.isin(days)
    # the STEM listening-group ids carry a round suffix ('_1' = first round)
    suffix = "_1" if prefix == "fisica_stem" else ""
    df.loc[mask, "participant_id"] = [
        f"{prefix}_{i:03d}{suffix}" for i in range(1, mask.sum() + 1)
    ]


def assign_id_by_time(df, start_time, end_time, prefix, start_idx=1):
    """Assign participant_id to liceo students (Q20 == '3') within a StartDate interval."""
    mask = (
        (df["StartDate"] >= start_time)
        & (df["StartDate"] < end_time)
        & (df["Q20"] == "3")
    )
    df.loc[mask, "participant_id"] = [
        f"{prefix}_{i:03d}" for i in range(start_idx, start_idx + mask.sum())
    ]
    return mask, start_idx + mask.sum()

In [17]:
data["StartDate"] = pd.to_datetime(data["StartDate"])

# on 2026-05-04 the survey was restructured: Q20 became studente/docente and the department
# question moved to Q21 (students only). before that date the department was in Q20.
AFFILIATION_CUTOFF = pd.Timestamp("2025-12-02")
pre_affiliation = data["StartDate"] < AFFILIATION_CUTOFF
post_affiliation = ~pre_affiliation

# --- old survey (department in Q20): 1 fisica / 2 psicologia / 3 liceo ---
assign_id_by_question(data, "1", "fisica", pre_affiliation)
assign_id_by_question(data, "2", "psicologia", pre_affiliation)
assign_id_by_question(data, "3", "liceo", pre_affiliation)

# interfaces study (single day) and STEM-anxiety listening group (multiple sessions)
assign_id_by_date(data, "2025-03-26", "interfacce")
assign_id_by_date(
    data,
    ["2025-05-21", "2025-05-29", "2025-06-17", "2025-11-10", "2026-01-08"],
    "fisica_stem",
)

# split the liceo students into scientifico / linguistico by session time slot
idx_sci = 1
mask_sci, idx_sci = assign_id_by_time(
    data, "2025-05-13 11:40:00", "2025-05-13 11:45:00", "liceo_sci", idx_sci
)
mask_lin, _ = assign_id_by_time(
    data, "2025-05-13 12:27:00", "2025-05-13 12:40:00", "liceo_lin", 1
)
# start the second scientifico session one number later, leaving a slot for the late participant below
mask_sci_2, idx_sci = assign_id_by_time(
    data, "2025-05-30 07:50:00", "2025-05-30 08:55:00", "liceo_sci", idx_sci + 1
)
# a lone liceo scientifico student took part a few days later -> fills the reserved slot
data.loc[65, "participant_id"] = f"liceo_sci_{mask_sci.sum() + 1:03d}"

# STEM group: participants who came back for later rounds of questionnaires
data.loc[
    data["StartDate"].dt.date == pd.to_datetime("2025-07-22").date(), "participant_id"
] = "fisica_stem_002_2"
data.loc[
    data["StartDate"].dt.date == pd.to_datetime("2025-09-29").date(), "participant_id"
] = "fisica_stem_001_2"
data.loc[
    data["StartDate"].dt.date == pd.to_datetime("2025-11-19").date(), "participant_id"
] = "fisica_stem_002_3"

# --- new survey (department in Q21, students only): 1 fisica_unitn / 2 fisica / 3 psicologia / 4 liceo ---
# fisica_unitn is new; the other groups continue the numbering of their old-survey counterparts
n_fisica_pre = int((pre_affiliation & (data["Q20"] == "1")).sum())
n_psico_pre = int((pre_affiliation & (data["Q20"] == "2")).sum())
n_liceo_pre = int((pre_affiliation & (data["Q20"] == "3")).sum())
assign_id_by_question(data, "1", "fisica_unitn", post_affiliation, idx="Q21")
assign_id_by_question(
    data, "2", "fisica", post_affiliation, start_idx=n_fisica_pre + 1, idx="Q21"
)
assign_id_by_question(
    data, "3", "psicologia", post_affiliation, start_idx=n_psico_pre + 1, idx="Q21"
)
assign_id_by_question(
    data, "4", "liceo", post_affiliation, start_idx=n_liceo_pre + 1, idx="Q21"
)

assign_id_by_question(
    data,
    "5",
    "altro dipartimento",
    post_affiliation,
    start_idx=n_liceo_pre + 1,
    idx="Q21",
)

# teachers: new-survey participants who skipped the department question, so their Q21 is NaN
is_teacher = post_affiliation & data["Q21"].isna()
data.loc[is_teacher, "participant_id"] = [
    f"teacher_{i:03d}" for i in range(1, is_teacher.sum() + 1)
]

print(f"Participants: {len(data['participant_id'])}")
print(
    data["participant_id"]
    .str.replace(r"_\d.*", "", regex=True)
    .value_counts()
    .to_string()
)

Participants: 154
participant_id
liceo_sci             43
teacher               30
interfacce            23
liceo_lin             19
fisica                18
fisica_unitn           8
fisica_stem            7
altro dipartimento     4
liceo                  2


## Associations & valence tasks

`get_associations` / `get_valences` do the heavy lifting; a few relabellings are applied afterwards.

In [18]:
associations_task_df = get_associations(data)

# the "Meditazione" slot was renamed "ChatGPT" from the revision date on (config labels it "ChatGPT");
# relabel everyone who took part before the cutoff back to "Meditazione"
chatgpt_ids = set(data.loc[data["StartDate"] >= CHATGPT_CUTOFF, "participant_id"])
relabel_meditazione = (
    associations_task_df["cue_word"] == "ChatGPT"
) & ~associations_task_df["participant_id"].isin(chatgpt_ids)
associations_task_df.loc[relabel_meditazione, "cue_word"] = "Meditazione"

# for the early 'interfacce' participants that slot was still "Counselling"
interfacce_ai = associations_task_df["participant_id"].astype(str).str.startswith(
    "interfacce"
) & (associations_task_df["cue_word"] == "Intelligenza_Artificiale")
associations_task_df.loc[interfacce_ai, "cue_word"] = "Counselling"

# lowercase all free text
associations_task_df = associations_task_df.map(
    lambda x: x.lower() if isinstance(x, str) else x
)

In [19]:
valence_task_df = get_valences(data)
associations_valence_df = associations_valences_df(
    associations_task_df, valence_task_df
)

### Missing-data filtering & export

In [22]:
# mean fraction of missing answers per participant
missing_fraction = (
    associations_valence_df.set_index("participant_id")
    .isna()
    .mean(axis=1)
    .groupby(level=0)
    .mean()
)

# drop participants missing more than a third of their data (STEM-group ones are always kept)
to_drop = missing_fraction[
    (missing_fraction > 1 / 3) & ~missing_fraction.index.str.startswith("fisica_stem")
].index
print("Participants to drop:", len(to_drop), list(to_drop))

associations_valence = associations_valence_df[
    ~associations_valence_df["participant_id"].isin(to_drop)
].reset_index(drop=True)
associations_valence.to_csv(
    f"{path_out}/associations_valence_task_2025.csv", index=False
)
associations_valence

Participants to drop: 15 ['altro dipartimento_066', 'fisica_001', 'fisica_002', 'fisica_004', 'fisica_006', 'fisica_008', 'fisica_009', 'fisica_019', 'fisica_023', 'fisica_unitn_001', 'liceo_lin_019', 'liceo_sci_015', 'teacher_004', 'teacher_022', 'teacher_030']


,participant_id,cue_word,valence_cue_word,association_1,valence_association_1,association_2,valence_association_2,association_3,valence_association_3
0,interfacce_001,scienza,3,lontana,2,NaN,<NA>,NaN,<NA>
1,interfacce_002,scienza,4,studio,4,esperimenti,3,laboratorio,2
2,interfacce_003,scienza,4,chimica,1,einstein,3,NaN,<NA>
3,interfacce_004,scienza,4,esperimenti,4,conoscienza,4,verde,3
4,interfacce_005,scienza,4,chimica,2,NaN,<NA>,NaN,<NA>
...,...,...,...,...,...,...,...,...,...
5833,fisica_unitn_006,vita,3,biologia,4,lunga,3,replicare,4
5834,teacher_028,vita,4,biologia,3,natura,4,mistero,3
5835,fisica_unitn_007,vita,3,morte,5,inferno,4,dante,5
5836,teacher_029,vita,3,salute,4,esperienza,5,durata,3


In [21]:
mask = associations_valence["participant_id"].str.startswith("fisica")
associations_valence.loc[mask, "participant_id"].nunique()

24

## Questionnaires

Every questionnaire block is extracted with a single helper, then filtered on `to_drop` and
exported.

In [23]:
# gender and math self-perception / grade
gender = (
    data["Q17"].replace({"1": "M", "2": "F", "4": "NB", "5": "NA"}).rename("gender")
)
percieved_abilities = data["Q16"].rename("percieved_abilities")

# math grade: prefer the university grade (Q20_1) over the liceo one (Q21_1)
math_grade_uni = data["Q20_1"].replace(31, "30L")
math_grade_liceo = data["Q21_1"].replace(11, "10L")
math_grade = math_grade_uni.combine_first(math_grade_liceo).rename("math_grade")

In [24]:
def extract_block(data, question_id, start, end, col_label):
    """Pull a questionnaire's item columns ('{question_id}_{i}') into a labelled block."""
    block = data[[f"{question_id}_{i}" for i in range(start, end)]].copy()
    block.columns = [f"{col_label}_{i}" for i in range(start, end)]
    return block


cfs_data = extract_block(data, "Q22#1", 1, 13, "cfs")  # Cognitive Flexibility Scale
stai_data = extract_block(data, "QID77", 1, 19, "state_anxiety")  # STAI: state anxiety
big5_data = extract_block(data, "QID84", 1, 11, "big_5")  # Big Five (10 items)

# MAS_IT (reduced): keep the original, non-contiguous item numbering
mas_it_data = extract_block(data, "Q21#1", 1, 19, "mas_it")
mas_it_items = [1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 18, 19, 20, 21, 22, 23]
mas_it_data.columns = [f"mas_it_{n}" for n in mas_it_items]

In [25]:
def export(frame, filename):
    """Drop the excluded participants and write the frame to CSV."""
    frame = frame[~frame["participant_id"].isin(to_drop)].reset_index(drop=True)
    frame.to_csv(f"{path_out}/{filename}", index=False)
    return frame


mas_it_df = export(
    pd.concat([data["participant_id"], mas_it_data], axis=1), "mas_it_2025.csv"
)

all_questionnaires = export(
    pd.concat(
        [
            data["participant_id"],
            gender,
            percieved_abilities,
            math_grade,
            cfs_data,
            stai_data,
            big5_data,
            mas_it_data,
        ],
        axis=1,
    ),
    "all_questionnaires_2025.csv",
)

## Free text

In [26]:
# lowercase, strip trailing punctuation/whitespace, then end with a single full stop
text = (data["Q18"].str.lower().str.rstrip(".!? \n\t") + ".").rename("text")
text_df = export(pd.concat([data["participant_id"], text], axis=1), "free_text.csv")